# Day 2 — Step 1: 데이터 수집

Upbit API를 통해 암호화폐 3종목의 OHLCV 데이터를 수집하고  
하나의 DataFrame으로 통합합니다.

---

## OHLCV 데이터란?

| 컬럼 | 영어 | 의미 |
|------|------|------|
| open | Open | 하루 중 **첫 번째** 거래 가격 (시가) |
| high | High | 하루 중 **가장 높은** 가격 (고가) |
| low | Low | 하루 중 **가장 낮은** 가격 (저가) |
| close | Close | 하루 중 **마지막** 거래 가격 (종가) |
| volume | Volume | 하루 동안 거래된 **총 수량** (거래량) |

---

## 분석 종목

| 시세 코드 | 이름 | 특징 |
|-----------|------|------|
| KRW-BTC | 비트코인 | 세계 최초의 암호화폐, 시장의 기준점 |
| KRW-ETH | 이더리움 | 스마트 계약 기능이 있는 2세대 코인 |
| KRW-SOL | 솔라나 | 거래 속도가 매우 빠른 코인 |

---
## 0. 라이브러리 불러오기

| 라이브러리 | 비유 | 역할 |
|-----------|------|------|
| pyupbit | 배달앱 | 업비트 서버에서 코인 가격 데이터를 받아다 줌 |
| pandas | 엑셀 | 데이터를 표(DataFrame) 형태로 다룸 |

In [ ]:
import pyupbit          # 업비트 API로 코인 가격 데이터 수집
import pandas as pd     # 데이터를 표(DataFrame)로 다루기
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 불러오기 완료!")

---
## 1. 수집 설정

수집할 종목과 기간을 변수로 관리합니다.  
나중에 종목을 추가하거나 기간을 바꿀 때 이 셀만 수정하면 됩니다.

In [ ]:
# ── 분석할 종목 리스트 ────────────────────────────────────────────
TICKERS = ["KRW-BTC", "KRW-ETH", "KRW-SOL"]

# ── 수집 기간 설정 ────────────────────────────────────────────────
# MA5(5일 이동평균) 계산 시 처음 4일은 NaN이 되므로
# 여유분을 포함해 60일치를 받은 뒤 최근 30일만 사용
FETCH_DAYS = 60   # API에서 받을 일수
USE_DAYS   = 30   # 실제 분석에 사용할 일수

print(f"분석 종목 : {TICKERS}")
print(f"수집 일수 : {FETCH_DAYS}일치 → 최근 {USE_DAYS}일 사용")

---
## 2. 데이터 수집 함수

### Long Format(긴 형식)으로 합치는 이유

```
Wide Format (넓은 형식)       Long Format (긴 형식) ← 우리가 쓰는 방식
-------------------------------  ----------------------------------
date    BTC_close  ETH_close      ticker    date        close
1월1일  89,000,000  3,100,000      KRW-BTC  2026-01-01  89,000,000
1월2일  90,000,000  3,050,000      KRW-BTC  2026-01-02  90,000,000
                                   KRW-ETH  2026-01-01   3,100,000
                                   KRW-ETH  2026-01-02   3,050,000
```

Long Format은 종목이 늘어나도 열(컬럼)을 추가할 필요 없이 **행(row)만 늘어납니다.**

In [ ]:
def collect_ohlcv(tickers, fetch_days, use_days):
    """
    여러 종목의 OHLCV 데이터를 수집하고 하나의 DataFrame으로 합칩니다.

    매개변수:
        tickers    : 종목 코드 리스트 (예: ["KRW-BTC", "KRW-ETH"])
        fetch_days : API에서 받아올 일수
        use_days   : 실제 사용할 최근 일수

    반환값:
        ticker, date, open, high, low, close, volume 컬럼을 가진 DataFrame
    """
    all_frames = []   # 종목별 데이터를 임시로 담을 리스트

    for ticker in tickers:
        print(f"  [{ticker}] 수집 중...")
        try:
            # pyupbit.get_ohlcv() : 업비트에서 일봉 데이터 수집
            # count=fetch_days    : 최근 n일치 요청
            df = pyupbit.get_ohlcv(ticker, count=fetch_days)

            if df is None or df.empty:
                print(f"  [{ticker}] 수집 실패, 건너뜀")
                continue

            # .tail(n) : 마지막 n행(= 가장 최근 n일)만 추출
            df = df.tail(use_days).copy()

            # 날짜 컬럼 추가: 인덱스(datetime)에서 날짜 부분만 뽑기
            df['date']   = pd.to_datetime(df.index.date)

            # 종목 코드 컬럼 추가: 합쳤을 때 어느 종목인지 구분용
            df['ticker'] = ticker

            # reset_index(drop=True) : 날짜 인덱스를 버리고 0, 1, 2... 로 초기화
            df = df.reset_index(drop=True)

            all_frames.append(df)
            print(f"  [{ticker}] 완료 ({len(df)}일치)")

        except Exception as e:
            print(f"  [{ticker}] 오류: {e}, 건너뜀")
            continue

    # pd.concat() : 여러 DataFrame을 세로로 쌓기
    # ignore_index=True : 합친 후 인덱스를 0, 1, 2... 로 새로 매기기
    combined_df = pd.concat(all_frames, axis=0, ignore_index=True)
    return combined_df

In [ ]:
# ── 데이터 수집 실행 ───────────────────────────────────────────────
print("=== 데이터 수집 시작 ===")
df_raw = collect_ohlcv(TICKERS, FETCH_DAYS, USE_DAYS)
print(f"\n=== 수집 완료: 총 {len(df_raw)}행 ===")

In [ ]:
# ── 필요한 컬럼만 추출 및 정렬 ────────────────────────────────────
columns_needed = ['ticker', 'date', 'open', 'high', 'low', 'close', 'volume']
df_raw = df_raw[columns_needed].copy()

# 종목별, 날짜 오름차순 정렬
df_raw = df_raw.sort_values(by=['ticker', 'date'], ascending=True)
df_raw = df_raw.reset_index(drop=True)

print("[수집된 데이터 미리보기 (상위 5행)]")
print(df_raw.head())
print()
print("[컬럼 정보]")
df_raw.info()

In [ ]:
# ── 다음 단계(02_preprocessing)로 전달할 데이터 저장 ─────────────
# df_raw를 CSV로 저장해두면 다음 노트북에서 바로 불러올 수 있음
# index=False : 행 번호(인덱스)는 저장 안 함
df_raw.to_csv('ohlcv_raw.csv', index=False)
print("ohlcv_raw.csv 저장 완료 → 02_preprocessing.ipynb에서 불러옵니다")